[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-usedcar.ipynb)

# Full Project: Used Car Price Explorer (Log-Transform Regression)

*AIBits Academy · Machine Learning End To End · Full Project*

301 Indian used-car listings, and why a car's price doesn't depreciate in straight lines — the one modelling choice that fixes a linear model's worst errors on old, high-mileage cars.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['used_car_cardekho.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the CarDekho listings** (301 used cars).

In [ ]:
df = pd.read_csv('used_car_cardekho.csv')
print(df.shape)
df.head()

> **Business Problem**
>
> A used-car marketplace (OLX/CarDekho-style) wants an instant "fair price" estimate a seller can see the moment they enter their car's year, mileage, fuel type, and condition — before any human valuation. The and projects are also regression problems, but neither has a target that shrinks toward zero on a curve the way a depreciating asset's price does — that specific shape is this project's focus.

> **Dataset**
>
> **301 used-car listings** from CarDekho.com — Year, Selling_Price (₹ lakhs), Present_Price (current ex-showroom price, ₹ lakhs), Kms_Driven, Fuel_Type, Seller_Type, Transmission, Owner. [Dataset source (CarDekho, via Kaggle) →](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)

## Step 1 — Feature Engineering: Age, Not Year

A car's *year* on its own has no meaningful linear relationship with price — it's really a proxy for *age*, and age is what actually drives depreciation. The listings span 2003–2018 (scraped in 2019), so `car_age = 2019 - Year`:

In [ ]:
df['car_age'] = 2019 - df['Year']
df = pd.get_dummies(df, columns=['Fuel_Type','Seller_Type','Transmission'], drop_first=True)
print(df[['Selling_Price','Present_Price','Kms_Driven','car_age']].describe().round(2))

## Step 2 — Baseline: OLS on Raw Selling_Price

An 80/20 split (241 train / 60 test), then the obvious first model — ordinary least squares regressed directly on the rupee price:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = df.drop(columns=['Selling_Price','Car_Name','Year'])
y = df['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_raw = LinearRegression().fit(X_train, y_train)
pred_raw = lr_raw.predict(X_test)
print(f"RMSE (lakhs): {np.sqrt(mean_squared_error(y_test, pred_raw)):.4f}")
print(f"R-squared:    {r2_score(y_test, pred_raw):.4f}")

An R² of 0.51 leaves roughly half the price variance unexplained — worth checking exactly *where* this model is going wrong before reaching for a more complex algorithm.

## Step 3 — Where the Raw Model Fails: Old Cars

Grouping test-set prediction errors by car age reveals a clear, systematic pattern — not random noise:

In [ ]:
errors = pd.DataFrame({'age': X_test.car_age, 'error': pred_raw - y_test})
errors['bucket'] = pd.cut(errors.age, [0,3,7,11,99], labels=['0-3y','4-7y','8-11y','12y+'])
print(errors.groupby('bucket', observed=True)['error'].agg(['count','mean']).round(2))

For cars 12+ years old, the model *underpredicts* price by an average of ₹3.57 lakhs — a huge error relative to those cars' actual low prices. The problem is structural: linear regression on raw price assumes each extra year of age subtracts a **constant rupee amount**, but real depreciation is **multiplicative** — a car loses roughly a constant *percentage* of its remaining value each year, not a constant rupee amount. Extrapolated far enough, the linear model's constant-subtraction assumption eventually predicts unrealistically low (or even negative) prices for very old cars, when in reality depreciation slows once a car is already cheap.

## Step 4 — The Fix: Regress on log(Price) Instead

Taking the log of price converts multiplicative depreciation into an additive relationship in log-space — exactly the kind of transformation a linear model is built to fit well. Predictions are exponentiated back to rupees for a fair RMSE comparison against Step 2:

In [ ]:
y_train_log, y_test_log = np.log(y_train), np.log(y_test)
lr_log = LinearRegression().fit(X_train, y_train_log)
pred_log = np.exp(lr_log.predict(X_test))   # back to rupees (lakhs)

print(f"RMSE (lakhs, after exp): {np.sqrt(mean_squared_error(y_test, pred_log)):.4f}")
print(f"R-squared (lakhs scale): {r2_score(y_test, pred_log):.4f}")

errors_log = pd.DataFrame({'age': X_test.car_age, 'error': pred_log - y_test.values})
errors_log['bucket'] = pd.cut(errors_log.age, [0,3,7,11,99], labels=['0-3y','4-7y','8-11y','12y+'])
print(errors_log.groupby('bucket', observed=True)['error'].agg(['count','mean']).round(2))

## Visualising the Depreciation-Curve Fix

The raw model's bias (blue) grows sharply more negative with age — systematically underpricing old cars. The log-price model's bias (green) stays close to zero across every age bucket:

> **⚠ The Fix Isn't Free — and That's the Honest Result**
>
> Log-transforming isn't a strict win in every bucket: the newest cars (0–3 years) actually get a slightly *worse* mean error under the log model (−1.04 vs. −0.38 lakhs). Log-space regression trades a little accuracy on already-easy, high-price/low-age cars for a large bias reduction on the hard, old/high-mileage cars — and because there are far more mid-to-old cars than brand-new ones in this dataset, that trade is a clear net win overall (RMSE 2.98 → 1.32 lakhs, R² 0.513 → 0.904). This is the same honest, non-cherry-picked reporting standard as the rest of this course's projects — a real result, not the tidier story a fabricated one would tell.

## Key Business Takeaways

- Car depreciation is multiplicative (constant % per year), not additive (constant ₹ per year) — the single reason a plain linear regression on raw price systematically fails on old cars.
- Log-transforming the target before fitting, then exponentiating predictions back, cuts test RMSE by 55.6% (₹2.98 lakhs → ₹1.32 lakhs) and lifts R² from 0.513 to 0.904 — with no change to the algorithm, only to which scale it's fit on.
- The improvement isn't uniform: it comes almost entirely from fixing old-car bias, at a small cost on the newest cars — a nuance worth surfacing to stakeholders rather than reporting only the aggregate RMSE.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Price by fuel type

`df` now has dummy columns for fuel type. Re-read the raw file into `raw` and store the mean `Selling_Price` per `Fuel_Type` in `by_fuel` (a Series).

In [ ]:
raw = pd.read_csv("used_car_cardekho.csv")
by_fuel = None   # TODO


In [ ]:
try:
    ref = raw.groupby("Fuel_Type")["Selling_Price"].mean()
    check("same means", abs(by_fuel - ref).max() < 1e-12)
    check("diesel is priciest", by_fuel.idxmax() == "Diesel")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
raw = pd.read_csv("used_car_cardekho.csv")
by_fuel = raw.groupby("Fuel_Type")["Selling_Price"].mean()

```

</details>

### Exercise 2 · Medium · Age and price

Store the correlation between `car_age` and `Selling_Price` in `corr_age` and the correlation between `car_age` and `np.log(Selling_Price)` in `corr_age_log`. Which is stronger in magnitude, and why?

In [ ]:
corr_age = corr_age_log = None   # TODO


In [ ]:
try:
    check("negative", corr_age < 0 and corr_age_log < 0)
    check("log-price correlates more strongly with age", abs(corr_age_log) > abs(corr_age))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
corr_age = float(df["car_age"].corr(df["Selling_Price"]))
corr_age_log = float(df["car_age"].corr(np.log(df["Selling_Price"])))

```

</details>

### Exercise 3 · Stretch · Percentage error of the two models

Compute the **mean absolute percentage error** of the raw-target model (`pred_raw`) and of the log-target model (`pred_log`) on `y_test`, in `mape_raw` and `mape_log` (as fractions). The log model should win.

In [ ]:
mape_raw = mape_log = None   # TODO


In [ ]:
try:
    check("log model has lower percentage error", mape_log < mape_raw)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
yt = y_test.values
mape_raw = float(np.mean(np.abs(pred_raw - yt) / yt))
mape_log = float(np.mean(np.abs(pred_log - yt) / yt))

```

A log-target model implicitly minimises *relative* error, which is why it repairs the cheap and very old cars the plain model mis-prices.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Used Car Price Explorer (Log-Transform Regression)**.*